# Using RandALO with CVXPY

RandALO provides a small modeling layer for losses and regularizers, plus helpers that build both a CVXPY problem and its prediction Jacobian.

In [ ]:
import cvxpy as cp
import numpy as np

import randalo

rng = np.random.default_rng(0)

In [ ]:
n, p = 100, 30
X = rng.standard_normal((n, p))
beta = np.zeros(p)
beta[0:3] = 1.0
beta[9:12] = 2.0
y = X @ beta + 0.1 * rng.standard_normal(n)

## Forming the model

RandALO includes mean-squared-error and logistic losses. Custom loss functions can subclass `randalo.Loss`.

In [ ]:
loss = randalo.MSELoss()

Regularizers can be composed and scaled by a `randalo.HyperParameter`. This example uses grouped L2 penalties.

In [ ]:
alpha = randalo.HyperParameter()
groups = (
    randalo.L2Regularizer(list(range(i, i + 3)))
    for i in range(0, p, 3)
)
regularizer = alpha * sum(groups)

In [ ]:
variable = cp.Variable(p)
problem, jac = randalo.gen_cvxpy_jacobian(
    loss, regularizer, X, variable, y
)

In [ ]:
alpha.value = 0.1
problem.solve()

alo = randalo.RandALO(
    loss=loss,
    jac=jac,
    y=y,
    y_hat=X @ variable.value,
)
alo.evaluate(loss, n_matvecs=20, subsets=10)